# Nexus / GAIA — Gemini Flash Colab Backend

Canonical Colab-side REST adapter for the SSH host named colab. The API key is read from Colab Secrets and is never embedded in this notebook or printed. The backend returns the raw Gemini REST JSON expected by the Nexus SSH provider.

In [ ]:
from google.colab import userdata
from pathlib import Path
import json, os, subprocess, hashlib

def secret(name, default=""):
    try:
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return default

api_key = secret("GEMINI_API_KEY")
model = secret("GEMINI_MODEL", "gemini-flash-latest")
if not api_key:
    raise RuntimeError("Set a rotated key in Colab Secrets as GEMINI_API_KEY before installing the backend.")

In [ ]:
config_dir = Path("/root/.config/nexus")
bin_dir = Path("/root/nexus/bin")
config_dir.mkdir(parents=True, exist_ok=True)
bin_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "gemini.json"
config_path.write_text(json.dumps({"GEMINI_API_KEY": api_key, "GEMINI_MODEL": model}))
os.chmod(config_path, 0o600)

backend = r'''#!/usr/bin/env python3
import json, os, sys, urllib.request, urllib.error
from pathlib import Path
config = json.loads(Path("/root/.config/nexus/gemini.json").read_text())
key = os.environ.get("GEMINI_API_KEY") or config.get("GEMINI_API_KEY", "")
model = os.environ.get("GEMINI_MODEL") or config.get("GEMINI_MODEL", "gemini-flash-latest")
if not key:
    print(json.dumps({"error": "GEMINI_API_KEY is not configured"}), file=sys.stderr)
    raise SystemExit(78)
prompt = sys.stdin.read().strip() if not sys.argv[1:] else " ".join(sys.argv[1:]).strip()
if not prompt:
    print(json.dumps({"error": "empty_prompt"}), file=sys.stderr)
    raise SystemExit(64)
url = "https://generativelanguage.googleapis.com/v1beta/models/" + model + ":generateContent"
body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode("utf-8")
req = urllib.request.Request(url, data=body, method="POST",
    headers={"Content-Type": "application/json", "X-goog-api-key": key})
try:
    with urllib.request.urlopen(req, timeout=120) as response:
        sys.stdout.write(response.read().decode("utf-8"))
except urllib.error.HTTPError as exc:
    sys.stderr.write(exc.read().decode("utf-8", errors="replace"))
    raise SystemExit(70)
'''
backend_path = bin_dir / "gemini-backend"
backend_path.write_text(backend)
os.chmod(backend_path, 0o700)
print("Installed:", backend_path)
print("Secret config mode:", oct(config_path.stat().st_mode & 0o777))

In [ ]:
test = subprocess.run(
    [str(backend_path)],
    input="Return exactly: NEXUS_COLAB_GEMINI_OK",
    text=True, capture_output=True, timeout=150,
)
print("exit:", test.returncode)
if test.returncode != 0:
    print(test.stderr[:1200])
    raise RuntimeError("Gemini backend smoke test failed")
payload = json.loads(test.stdout)
text = "".join(
    part.get("text", "")
    for part in payload.get("candidates", [{}])[0].get("content", {}).get("parts", [])
    if isinstance(part, dict)
).strip()
print("model:", model)
print("response_sha256:", hashlib.sha256(test.stdout.encode("utf-8")).hexdigest())
print("response:", text)
assert "NEXUS_COLAB_GEMINI_OK" in text

SSH configuration:

    Host colab
      User root
      ProxyCommand colab ssh --proxy-mode -s colab
      StrictHostKeyChecking no
      UserKnownHostsFile /dev/null
      RequestTTY no
      RemoteCommand cd /content 2>/dev/null && exec /root/nexus/bin/gemini-backend

No credential is stored in the repository. The runtime config is ephemeral and owner-readable only.